# The 95 GeV excess: what the SM has to say about a second scalar

Three searches see small excesses at similar masses: CMS diphoton at 95.4 GeV, CMS ditau at
95 to 100 GeV and LEP $b\bar b$ near 98 GeV. None is significant on its own, and ATLAS sees much
less in diphoton. This notebook does the first two rungs of the ladder for this record (A1):

- it proves, with the SM taken from feynlag-models, that the SM has room for exactly **one**
  physical scalar, so it predicts no resonant signal at 95 GeV;
- it puts the ditau and diphoton channels on **one scale**, a signal strength $\mu$, so that a
  candidate model can later be confronted with both.

It does not fit anything and it does not claim the channels agree; that is A3 work.

Each section runs four moves: **set up**, **collect**, **recognise**, **check**. Before the cell
that settles each question, a blockquote asks you to commit to an expectation.

In [1]:
import math
import sys
from pathlib import Path

import sympy as sp
from IPython.display import Markdown, Math, display

# nbclient/nbmake set the kernel's cwd to this notebook's own directory; make sure it (and the
# repo root two levels up) are importable regardless of how the kernel was launched.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parents[1]
for p in (NB_DIR, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from feynlag_anomalies.latex import latex
from feynlag_anomalies.registry import load as load_anomaly
from anomalies.higgs_95gev import checkpoints_def as cd

sp.init_printing(use_latex="mathjax", latex_printer=latex)


def ok(msg):
    print("✓", msg)


anomaly = load_anomaly("higgs_95gev")
obs = {o.name: o for o in anomaly.observables}
print(f"{anomaly.title}: status {anomaly.status} ({anomaly.status_date}), maturity {anomaly.maturity}")

~95 GeV diphoton/ditau/bb excess: status hint (2026-10-15), maturity A1


---
## 0. What is measured: three small excesses, two different units

The record holds what each experiment printed, read from the papers and CMS's own supplementary
material *(cited, see `anomaly.yaml` `sources`)*.

> **Before running the next cell.** CMS diphoton prints a signal strength $\mu$, a pure number.
> CMS ditau prints $\sigma \times B$ in pb. Can you compare $\mu_{\gamma\gamma} = 0.33$ with
> $\sigma B_{\tau\tau} = 7.8$ pb? What extra number would you need?

In [2]:
# ---- MOVE 1: set up.  The observables, straight from anomaly.yaml. ----
fmt = lambda x: x if isinstance(x, str) else f"{x:.4g}"
rows = ["| observable | value | ± (stat. col.) | ± (sys. col.) | units |", "|---|---|---|---|---|"]
rows += [f"| {o.name} | {fmt(o.value)} | {fmt(o.stat_uncertainty)} | "
         f"{'—' if o.sys_uncertainty is None else fmt(o.sys_uncertainty)} | {o.units} |"
         for o in anomaly.observables]
display(Markdown("\n".join(rows)))
sig = anomaly.significance
print(f"diphoton significance: {sig.local} sigma local, {sig.global_} sigma global")
units = {o.units for o in anomaly.observables}
assert "pb" in units and len(units) > 1
ok("the channels are recorded in different units: pb for ditau, a ratio for diphoton")

| observable | value | ± (stat. col.) | ± (sys. col.) | units |
|---|---|---|---|---|
| sigma(gg -> phi) x B(phi -> tau tau) at m_phi = 95 GeV (CMS ditau) | 7.8 | 3.5 | — | pb |
| signal strength mu at 95.4 GeV (CMS diphoton, best fit) | 0.33 | 0.155 | — | ratio to SM-like Higgs cross section |
| signal strength mu_tautau at 95 GeV (CMS ditau, normalised to SM-like ggF x B) | 1.374 | 0.616 | 0.159 | ratio to SM-like Higgs ggF cross section x B(tau tau) |
| signal strength mu near 98 GeV (LEP bb-bar, best fit) | TODO_VERIFY | TODO_VERIFY | — | ratio to SM Higgs cross section |

diphoton significance: 2.9 sigma local, 1.3 sigma global
✓ the channels are recorded in different units: pb for ditau, a ratio for diphoton


**Read it.** The diphoton number is already a ratio to an SM-like Higgs boson of the same mass;
the ditau one is a cross section. They cannot be compared until the ditau number is divided by
what an SM-like Higgs boson at 95 GeV would give in the same channel. That denominator is the
subject of section 2. The fourth row is the result of section 2, stored in the record and
recomputed below; the LEP row is still `TODO_VERIFY`, because the LEP paper prints no signal
strength.

First, the question that makes this an anomaly: is there any room in the SM for a scalar at
95 GeV?

---
## 1. The SM has one scalar, and it is not at 95 GeV

The SM's scalar sector is one complex doublet $H = (G^+, H^0)$. The model comes from
feynlag-models (`sm`, maturity L3) through the registry; nothing is declared here.

> **Before running the next cells.** A complex doublet has four real fields. After electroweak
> symmetry breaking, $W^\pm$ and $Z$ become massive. How many physical scalars are left, and is
> there any free choice in that number?

In [3]:
# ---- MOVE 1: set up.  The SM from the registry, and its doublet. ----
from feynlag_models.registry import build

sm = build("sm")
ew = sm.extra["ew"]
display(Math(r"H = \begin{pmatrix}" + r"\\".join(latex(c) for c in ew.H.components) + r"\end{pmatrix}"))
n_dof = 2 * len(ew.H.components)
print(f"real degrees of freedom in H: {n_dof}")

<IPython.core.display.Math object>

real degrees of freedom in H: 4


In [4]:
# ---- MOVE 2: collect.  The model's own scalar mass matrices after EWSB. ----
b = sm.bosons
M_neutral = sm.model.mass_matrix([b["h"], b["G0"]])
M_charged = sm.model.mass_matrix([b["Gp"]], charged=True)
display(Math(r"M^2_{(" + latex(b["h"]) + "," + latex(b["G0"]) + r")} = " + latex(M_neutral)))
display(Math(r"M^2_{(" + latex(b["Gp"]) + r")} = " + latex(M_charged)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

**Recognise.** The neutral block is diagonal: one eigenvalue $2\lambda v^2$, one zero. The
charged block is zero. The zeros are the would-be Goldstones $G^0$, $G^\pm$, three real fields,
which become the longitudinal $Z$ and $W^\pm$. So $4 - 3 = 1$ physical scalar, $h$, with
$m_h^2 = 2\lambda v^2$ *(computed; the same result is the model's key result
`test_tadpole_and_higgs_mass` in feynlag-models)*. There is no free choice: the count is fixed
by the field content, and only the mass, through $\lambda$, is an input.

In [5]:
# ---- MOVE 4: check.  Count eigenvalues, against the checkpoint. ----
check_1 = cd.make_check_1(sm)
eig_n = [e for e, m in M_neutral.eigenvals().items() for _ in range(m)]
eig_c = [e for e, m in M_charged.eigenvals().items() for _ in range(m)]
massive = [e for e in eig_n if sp.simplify(e) != 0]
n_goldstone = (len(eig_n) - len(massive)) + 2 * sum(1 for e in eig_c if sp.simplify(e) == 0)
answer_1 = len(massive)
assert n_goldstone == len(sm.goldstones) == 3
assert n_dof - n_goldstone == answer_1
assert sp.simplify(massive[0] - 2 * ew.lam.s * ew.v.s**2) == 0
assert check_1(answer_1)
ok("one physical neutral scalar, m_h^2 = 2 lambda v^2, three Goldstones eaten")

[step_1] correct -- got 1
✓ one physical neutral scalar, m_h^2 = 2 lambda v^2, three Goldstones eaten


Now put in numbers. The model's benchmark sets $\lambda$ from its input $m_h$; the count
above says that this one input is all the SM has.

> **Before running the next cell.** With one scalar whose mass is chosen to be the observed
> Higgs boson, what does the SM predict for a resonance at 95 GeV?

In [6]:
# ---- MOVE 4: check.  Every physical SM scalar at the benchmark, against the 95 GeV window. ----
values = sm.values()
masses = [math.sqrt(float(m2.subs(values))) for m2 in massive]
lo, hi = cd.WINDOW_GEV   # the mass window of the excesses (CMS gamma gamma, CMS tau tau, LEP)
in_window = [m for m in masses if lo <= m <= hi]
print(f"SM scalar masses at the benchmark: {[f'{m:.2f} GeV' for m in masses]} (input MH = {sm.benchmark['MH']})")
assert in_window == []
ok(f"no SM scalar in {lo:.0f}-{hi:.0f} GeV: the SM predicts zero resonant signal there")

SM scalar masses at the benchmark: ['125.00 GeV'] (input MH = 125.0)
✓ no SM scalar in 90-100 GeV: the SM predicts zero resonant signal there


**A trap worth a cell.** The SM *does* have a state in that window: the $Z$ boson.

In [7]:
# ---- peek: the Z mass at the benchmark. ----
m_Z = sm.params.numeric()[sm.extra["MZ"].symbol]
display(Math(r"m_Z = " + latex(sm.extra["MZ"].expr) + rf" = {m_Z:.2f}\ \text{{GeV}}"))

<IPython.core.display.Math object>

The $Z$ sits at 91 GeV, inside the window, but it is spin 1. A spin-1 particle cannot decay to
two photons (the Landau–Yang theorem) *(cited: L. D. Landau, Dokl. Akad. Nauk SSSR 60 (1948)
207; C. N. Yang, Phys. Rev. 77 (1950) 242)*, so it cannot be the diphoton peak. In the ditau
and $b\bar b$ channels it is a known background, already part of the SM expectation the
experiments subtract.

**The judgment to keep:** the SM's prediction at 95 GeV is a *structural* zero. It comes from
the field content, one doublet, not from a parameter choice. The protection is `field_content`
*(see [`docs/protections.md`](../../docs/protections.md))*: a second scalar needs a new field.
That is why a real resonance here would be new physics, and why the minimal models to try add
one scalar field (a singlet or a second doublet).

---
## 2. Ditau and diphoton on one scale

A signal strength divides what is seen by what an SM-like Higgs boson of the same mass would
give in the same channel:

$$\mu_{\tau\tau} = \frac{[\sigma(gg\to\phi)\,B(\phi\to\tau\tau)]_{\rm obs}}
{\sigma_{\rm ggF}^{\rm SM\text{-}like}(95\ \text{GeV}, 13\ \text{TeV})\;
B^{\rm SM\text{-}like}(H\to\tau\tau, 95\ \text{GeV})} .$$

"SM-like" means a hypothetical Higgs boson with SM couplings and a mass of 95 GeV. Its numbers
are tabulated by the LHC Higgs Working Group *(cited, stored in
`sm_prediction.sm_reference_values`)*.

> **Before running the next cells.** A 95 GeV SM-like Higgs boson is produced more often than
> the 125 GeV one (gluon luminosity is larger at lower mass). Is $\mu_{\tau\tau}$ then larger or
> smaller than 1, given $\sigma B_{\rm obs} = 7.8$ pb? Guess the size of the denominator first.

In [8]:
# ---- MOVE 1: set up.  The SM-like inputs and their sources, from the record. ----
ref = anomaly.sm_prediction.sm_reference_values
rows = ["| input | value |", "|---|---|"] + [f"| `{k}` | {v:g} |" for k, v in ref.items()]
display(Markdown("\n".join(rows)))
for s in anomaly.sources:
    if "LHC Higgs WG" in s.identifier:
        print(f"source: {s.identifier.split(' (')[0]} -- consulted {s.consulted_on}")

| input | value |
|---|---|
| `sigma_ggF_95GeV_13TeV_pb` | 68.25 |
| `sigma_ggF_95GeV_13TeV_scale_up_pct` | 8.6 |
| `sigma_ggF_95GeV_13TeV_scale_down_pct` | 8.9 |
| `sigma_ggF_95GeV_13TeV_pdf_alphas_pct` | 3.3 |
| `BR_tautau_95GeV` | 0.0832 |
| `BR_tautau_95GeV_up_pct` | 6.94 |
| `BR_tautau_95GeV_down_pct` | 6.77 |

source: LHC Higgs WG twiki LHCPhysics/CERNYellowReportPageBSMAt13TeV -- consulted 2026-10-16
source: LHC Higgs WG twiki LHCPhysics/CERNYellowReportPageBR2014 -- consulted 2026-10-16


In [9]:
# ---- MOVE 2: collect.  The ratio, by hand from the record. ----
ditau = obs["sigma(gg -> phi) x B(phi -> tau tau) at m_phi = 95 GeV (CMS ditau)"]
denom = ref["sigma_ggF_95GeV_13TeV_pb"] * ref["BR_tautau_95GeV"]
answer_2 = ditau.value / denom
exp_err = ditau.stat_uncertainty / denom
print(f"SM-like sigma_ggF x B(tau tau) at 95 GeV = {denom:.3f} pb")
print(f"mu_tautau = {ditau.value} pb / {denom:.3f} pb = {answer_2:.3f} +- {exp_err:.3f} (exp)")

SM-like sigma_ggF x B(tau tau) at 95 GeV = 5.678 pb
mu_tautau = 7.8 pb / 5.678 pb = 1.374 +- 0.616 (exp)


**Recognise.** The denominator is about 5.7 pb, so 7.8 pb is a signal strength a little above 1.
The experimental error, the symmetrised CMS error propagated linearly, is about 45 % of the value.

The theory error on the denominator is separate: the ggF scale variation, its PDF+$\alpha_s$
error and the $B(\tau\tau)$ error, each symmetrised and added in quadrature *(decision, see
`decisions.md`)*. It is kept apart from the experimental error so that a later fit can treat it
as a correlated nuisance.

In [10]:
# ---- MOVE 4: check.  Against the checkpoint and the row stored in anomaly.yaml. ----
check_2 = cd.make_check_2(anomaly)
assert check_2(answer_2)
scale = (ref["sigma_ggF_95GeV_13TeV_scale_up_pct"] + ref["sigma_ggF_95GeV_13TeV_scale_down_pct"]) / 2
br = (ref["BR_tautau_95GeV_up_pct"] + ref["BR_tautau_95GeV_down_pct"]) / 2
th_err = answer_2 * math.sqrt(scale**2 + ref["sigma_ggF_95GeV_13TeV_pdf_alphas_pct"] ** 2 + br**2) / 100
row = obs["signal strength mu_tautau at 95 GeV (CMS ditau, normalised to SM-like ggF x B)"]
assert abs(row.value - answer_2) < 5e-4 and abs(row.stat_uncertainty - exp_err) < 5e-4
assert abs(row.sys_uncertainty - th_err) < 5e-4
ok(f"mu_tautau = {answer_2:.2f} +- {exp_err:.2f} (exp) +- {th_err:.2f} (theory) matches the record")

[step_2] correct -- got 1.37363
✓ mu_tautau = 1.37 +- 0.62 (exp) +- 0.16 (theory) matches the record


Now the two signal strengths side by side.

> **Before running the next cell.** Both numbers are now ratios to an SM-like Higgs boson. Name
> one reason why they need not be equal even for a single new scalar.

In [11]:
# ---- MOVE 3: recognise.  Side by side, without a verdict. ----
gg = obs["signal strength mu at 95.4 GeV (CMS diphoton, best fit)"]
display(Markdown("\n".join([
    "| channel | production | $\\mu$ | ± exp. | ± theory |", "|---|---|---|---|---|",
    f"| CMS $\\gamma\\gamma$, 95.4 GeV | all modes | {gg.value:.2f} | {gg.stat_uncertainty:.3f} (symmetrised) | — |",
    f"| CMS $\\tau\\tau$, 95 GeV | $gg\\to\\phi$ | {answer_2:.2f} | {exp_err:.2f} | {th_err:.2f} |",
])))

| channel | production | $\mu$ | ± exp. | ± theory |
|---|---|---|---|---|
| CMS $\gamma\gamma$, 95.4 GeV | all modes | 0.33 | 0.155 (symmetrised) | — |
| CMS $\tau\tau$, 95 GeV | $gg\to\phi$ | 1.37 | 0.62 | 0.16 |

**Read it, and stop there.** These are two numbers on one scale; nothing here tests whether they
are compatible. That needs a likelihood with correlated theory errors and the asymmetric
experimental errors, which is the stage-1 fit at A3.

**Caveats** that any comparison must carry:

- **Production differs.** Ditau is gluon fusion only; the diphoton $\mu$ is for the combined
  production modes. A model with non-SM ratios of production modes moves them differently.
- **Decay differs.** A scalar whose couplings are not a common rescaling of the SM ones changes
  $B(\gamma\gamma)$ and $B(\tau\tau)$ differently, so $\mu_{\gamma\gamma} \neq \mu_{\tau\tau}$
  is allowed in principle.
- **The masses are not identical.** Ditau peaks at 100 GeV (3.1σ local) and is evaluated here at
  95 GeV; diphoton is at 95.4 GeV.
- **Inputs from two reports.** The cross section is from CERN Report 4, the branching ratio from
  CERN Report 3, the only LHC Higgs WG table with an SM-like $B$ at 95 GeV. A cross-check from
  the Report 4 partial widths at 95 GeV agrees to about 1 % *(computed, see `decisions.md`)*.

**The judgment to keep:** a channel only enters a fit once it is on the same scale as the others.
Converting a cross section into a signal strength makes it comparable, and it also brings in a
theory error that comes from the SM-like denominator, not from the experiment.

---
## 3. Recap

**The numbers**, each one produced by a cell above:

In [12]:
summary = [
    ("1", "the SM has one physical scalar after EWSB", f"{answer_1} ($m_h^2 = 2\\lambda v^2$; 4 dof − 3 Goldstones)", "computed"),
    ("1", "no SM scalar in 90–100 GeV", f"masses at benchmark: {', '.join(f'{m:.0f}' for m in masses)} GeV", "computed"),
    ("2", "SM-like $\\sigma_{\\rm ggF} B_{\\tau\\tau}$ at 95 GeV, 13 TeV", f"{denom:.2f} pb", "computed (inputs cited)"),
    ("2", "CMS ditau as a signal strength", f"$\\mu_{{\\tau\\tau}}$ = {answer_2:.2f} ± {exp_err:.2f} (exp) ± {th_err:.2f} (th)", "computed"),
    ("0", "CMS diphoton signal strength", f"$\\mu_{{\\gamma\\gamma}}$ = {gg.value:.2f} ± {gg.stat_uncertainty:.3f} (symmetrised)", "cited"),
]
rows = ["| § | result | key number | provenance |", "|---|---|---|---|"]
rows += [f"| {s} | {r} | {n} | {p} |" for s, r, n, p in summary]
display(Markdown("\n".join(rows)))

| § | result | key number | provenance |
|---|---|---|---|
| 1 | the SM has one physical scalar after EWSB | 1 ($m_h^2 = 2\lambda v^2$; 4 dof − 3 Goldstones) | computed |
| 1 | no SM scalar in 90–100 GeV | masses at benchmark: 125 GeV | computed |
| 2 | SM-like $\sigma_{\rm ggF} B_{\tau\tau}$ at 95 GeV, 13 TeV | 5.68 pb | computed (inputs cited) |
| 2 | CMS ditau as a signal strength | $\mu_{\tau\tau}$ = 1.37 ± 0.62 (exp) ± 0.16 (th) | computed |
| 0 | CMS diphoton signal strength | $\mu_{\gamma\gamma}$ = 0.33 ± 0.155 (symmetrised) | cited |

**The mechanics.**

0. The record holds what each experiment printed, in its own units: a ratio for diphoton, a cross
   section for ditau, nothing yet for LEP.
1. The SM from feynlag-models has one doublet: four real fields, three eaten by $W^\pm$, $Z$,
   one physical scalar with $m_h^2 = 2\lambda v^2$. Its mass is an input, so the SM predicts no
   second resonance anywhere, in particular none at 95 GeV. The protection is `field_content`.
2. Dividing the ditau $\sigma B$ by the SM-like $\sigma_{\rm ggF} B_{\tau\tau}$ at 95 GeV puts it
   on the diphoton scale, with a theory error kept apart from the experimental one.

**The tools, in the order you should reach for them.**

| question | tool |
|---|---|
| how many physical scalars? | `bundle.model.mass_matrix` on the neutral and charged blocks, then count nonzero eigenvalues |
| what is their mass? | the same eigenvalues, `subs(bundle.values())` at the benchmark |
| a channel in pb, the others in $\mu$? | divide by the SM-like $\sigma \times B$ from the LHC Higgs WG tables (`solutions/step_2.py`) |
| inputs consumed by code | `anomaly.sm_prediction.sm_reference_values`, rejected if `TODO_VERIFY` |

**The traps**, each of which a cell above turned into a check or a caveat:

- **The $Z$ is in the window.** It is spin 1, cannot give a diphoton peak (Landau–Yang), and is
  already part of the background.
- **"SM-like" is a hypothetical boson**, the SM Higgs with its mass moved to 95 GeV. It is a
  normalisation, not an SM prediction for a signal: the SM predicts zero.
- **Equal units do not mean comparable channels.** Production and decay differ between ditau and
  diphoton.
- **Two error types.** Experimental and theory normalisation errors are kept apart for the fit.

**Next steps (A2).** Identify the lowest-dimension EFT description of a light scalar coupling
to the SM, and reference minimal candidate models by `model_id`. Two in feynlag-models are at
L3: `sm_singlet_z2` (a real singlet mixing with the Higgs) and `thdm_type2` (a second doublet).
In the singlet model every coupling of the lighter state is the SM one times a mixing factor
*(cited: model key result `test_coupling_rescaling`)*, which suggests the question to answer
there: can one mixing angle give both $\mu_{\gamma\gamma}$ and $\mu_{\tau\tau}$? The LEP
$b\bar b$ signal strength is still to be found in a primary source.